# Cloud Provider Analytics: ejecución en Google Colab

Corre el pipeline (Landing → Bronze → … → Serving) en Colab, sin Docker.

**Credenciales:** se leen de **Colab Secrets** nunca se escriben en el notebook ni en Drive.
Cargar estos secrets y habilitarles el acceso a este notebook:

| Secret | Valor |
|---|---|
| `ASTRA_DB_APPLICATION_TOKEN` | Token de AstraDB (`AstraCS:...`) |
| `ASTRA_DB_API_ENDPOINT` | `https://<db_id>-<region>.apps.astra.datastax.com` |

El secure connect bundle se descarga solo con esos dos valores y queda en el disco de la VM
(`/content/tpe-big-data/secrets/`), que se borra al cerrar la sesión. No va a Drive.

## 1. Código y dependencias

In [ ]:
import os, sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/VieraFederico/tpe-big-data.git"
BRANCH = "main"
FORCE_CLONE = False
LOCAL_CLONE_DIR = Path.home() / "tpe-big-data"

IN_COLAB = "google.colab" in sys.modules
START_DIR = globals().get("START_DIR", Path.cwd().resolve())

if IN_COLAB or FORCE_CLONE:
    target = Path("/content/tpe-big-data") if IN_COLAB else LOCAL_CLONE_DIR.resolve()
    if not IN_COLAB and START_DIR.is_relative_to(target):
        raise RuntimeError(f"El notebook está dentro de {target}; elegí otro LOCAL_CLONE_DIR.")
    os.chdir(target.parent)
    shutil.rmtree(target, ignore_errors=True)
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} "{target}"
    os.chdir(target)
else:
    root = START_DIR
    while not (root / "config/requirements.txt").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

print("Trabajando en:", Path.cwd())

%pip install -q --only-binary=:all: cassandra-driver
reqs = [l.strip() for l in Path("config/requirements.txt").read_text().splitlines()
        if l.strip() and not l.strip().startswith(("#", "cassandra-driver"))]
%pip install -q {" ".join(reqs)}

import cassandra, pyspark
print("Colab" if IN_COLAB else "Local", "| cassandra-driver", cassandra.__version__, "| pyspark", pyspark.__version__)

In [ ]:
import shutil, subprocess
if shutil.which("java") is None:
    !apt-get -qq install -y openjdk-17-jre-headless > /dev/null
!java -version 2>&1 | head -1

## 2. Configuración

In [ ]:
import os
from google.colab import userdata

os.environ["SERVING_TARGET"] = "astra"
os.environ["CASSANDRA_KEYSPACE"] = "cloud_analytics"
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = userdata.get("ASTRA_DB_APPLICATION_TOKEN")
os.environ["ASTRA_DB_API_ENDPOINT"] = userdata.get("ASTRA_DB_API_ENDPOINT")
os.environ["DATALAKE_PATH"] = "/content/tpe-big-data/data/datalake"

print("SERVING_TARGET =", os.environ["SERVING_TARGET"])
print("DATALAKE_PATH  =", os.environ["DATALAKE_PATH"])
print("Token cargado  =", bool(os.environ["ASTRA_DB_APPLICATION_TOKEN"]))

## 3. Pipeline

In [ ]:
!python -m src.pipeline --list

In [ ]:
!python -m src.pipeline --steps bronze_batch,bronze_stream

## 4. Serving (AstraDB)

In [ ]:
!python -m src.serving.check_connection

## 5. Evidencias

Los logs quedan en `evidence/logs/`. Descargarlos para sumarlos al repo.

In [ ]:
!ls -lh evidence/logs/
!du -sh data/datalake/*/ 2>/dev/null

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/evidence_logs", "zip", "evidence/logs")
files.download("/content/evidence_logs.zip")